# Sample notebook for testing the ingestion module

This notebook demonstrates how to import and run the document ingestion workflow from the RAG package.

In [ ]:
import importlib
import logging
import os
import subprocess
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'services').exists() and (candidate / 'requirements.txt').exists():
            return candidate
    return start


def ensure_dependencies() -> None:
    req_file = repo_root / 'requirements.txt'
    if req_file.exists():
        print('Installing project dependencies from requirements.txt...')
        subprocess.check_call([
            sys.executable,
            '-m',
            'pip',
            'install',
            '--upgrade',
            '--prefer-binary',
            '-r',
            str(req_file),
        ])
        print('Dependencies installed.')
    else:
        print('requirements.txt not found; skipping dependency install.')


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

ensure_dependencies()

venv_python = repo_root / '.venv' / 'Scripts' / 'python.exe'
if venv_python.exists():
    os.environ['VIRTUAL_ENV'] = str(repo_root / '.venv')
    os.environ['PATH'] = f"{repo_root / '.venv' / 'Scripts'};{os.environ.get('PATH', '')}"

# Make notebook output show logging.info messages as well as print() output.
logging.basicConfig(level=logging.INFO, format='%(levelname)s:%(name)s:%(message)s', stream=sys.stdout)
logging.basicConfig(level=logging.WARNING, format='%(levelname)s:%(name)s:%(message)s', stream=sys.stdout)
logging.basicConfig(level=logging.ERROR, format='%(levelname)s:%(name)s:%(message)s', stream=sys.stdout)
logging.basicConfig(level=logging.DEBUG, format='%(levelname)s:%(name)s:%(message)s', stream=sys.stdout)
logging.getLogger().setLevel(logging.INFO)
logging.getLogger().setLevel(logging.WARNING)
logging.getLogger().setLevel(logging.ERROR)
#logging.getLogger().setLevel(logging.DEBUG)

import services.rag.models as models_module
import services.rag.embeddings as embeddings_module
import services.rag.injest as injest_module
import services.rag.visualization as visualization_module

for module in [models_module, embeddings_module, injest_module, visualization_module]:
    try:
        importlib.reload(module)
    except Exception as exc:
        print(f'Reload warning for {module.__name__}: {exc}')

Injestor = injest_module.Injestor
ChunkEmbeddingAnalyzer = visualization_module.ChunkEmbeddingAnalyzer

print('Imported Injestor and ChunkEmbeddingAnalyzer successfully')
print(f'Notebook working directory: {repo_root}')

In [ ]:
import asyncio
from pathlib import Path

sample_dir = repo_root / 'notebooks' / 'sample_docs'
sample_dir.mkdir(parents=True, exist_ok=True)

sample_file = sample_dir / 'sample.txt'
sample_file.write_text(
    'The quick brown fox jumps over the lazy dog. ' * 20,
    encoding='utf-8',
)

print(f'Sample file created at: {sample_file}')

In [ ]:
async def run_ingestion():
    financeDocs = repo_root / 'notebooks' / 'finance_docs'
    use_external = os.getenv("OPENAI_API_KEY") is not None and os.getenv("OPENAI_API_KEY", "").strip() != ""
    if use_external:
        ingestor = Injestor(
            source_dir=financeDocs,
            model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
            provider="openai",
            use_external_model=True,
        )
        print("Using external OpenAI-compatible model via LiteLLM")
    else:
        ingestor = Injestor(source_dir=financeDocs, model='gemma3:270m')
        print("Using local Ollama model")

    print(f'Ingesting documents from: {financeDocs}')
    results = await ingestor.ingest_documents()
    return results

results = await run_ingestion()
print(f'Processed {len(results)} document(s)')
for result in results:
    print(f"Document: {result.document_name}")
    print(f"Chunks: {result.chunk_count}")
    for chunk in result.chunks[:3]:
        print(' -', chunk.headline, '|', chunk.summary, '| words:', chunk.word_count, '| text:', chunk.chunk_text[:50], '...')
    print()

In [ ]:
for result in results:
    print(f"Document: {result.document_name}")
    print(f"Chunks: {result.chunk_count}")
    for chunk in result.chunks:
        print(' -', chunk.headline, '|', chunk.summary, '| words:', chunk.word_count, '| text:', chunk.chunk_text, '...')
    print()

from services.rag.visualization import ChunkEmbeddingAnalyzer

collection_name = "sample_docs_chunks"
persist_directory = repo_root / '.chroma'
analyzer = ChunkEmbeddingAnalyzer(
    persist_directory=persist_directory,
    collection_name=collection_name,
)

print(f'Persist directory: {persist_directory}')
print(f'Analyzing embeddings in Chroma collection: {collection_name}')
print('Available collections:', [c.name for c in analyzer._client.list_collections()])

summary = analyzer.build_summary()
print('Embedding summary:')
for key, value in summary.items():
    print(f' - {key}: {value}')

if summary.get('record_count', 0) > 0:
    print('Generating 2D visualization...')
    analyzer.visualize_2d(output_path=repo_root / 'notebooks' / 'chunk_embeddings_2d.html', show=False)

    print('Generating 3D visualization...')
    analyzer.visualize_3d(output_path=repo_root / 'notebooks' / 'chunk_embeddings_3d.html', show=False)

    print('Visualization files created in notebooks/ folder.')
else:
    print('No embeddings were found in the Chroma collection yet; rerun ingestion first.')

In [ ]:
#inspect chroma vectors

from cmath import sqrt
from pathlib import Path

import chromadb
from chromadb.config import Settings

DB_PATH = Path("C:\\Rohit\\Trainings\\repo\\FinancialAnalystCopilot\\database\\chroma").resolve()

print(f"Opening Chroma database at: {DB_PATH}")

client = chromadb.PersistentClient(
    path=str(DB_PATH),
    settings=Settings(anonymized_telemetry=False),
)

collections = client.list_collections()
collection_names = [collection.name for collection in collections]

print(f"Available collections: {collection_names}")

for collection in collections:
    collection = client.get_collection(name=collection.name)

    print(f"Collection: {collection.name}")
    print(f"Number of records: {collection.count()}")

    results = collection.get(
        limit=5,
        include=["documents", "metadatas", "embeddings"],
    )

    embeddings = results.get("embeddings")
    documents = results.get("documents")
    metadatas = results.get("metadatas")

    for index, record_id in enumerate(results["ids"]):
        vector = embeddings[index] if embeddings is not None else None
        document = documents[index] if documents is not None else None
        metadata = metadatas[index] if metadatas is not None else None

        print("\n" + "=" * 80)
        print(f"ID: {record_id}")
        print(f"Metadata: {metadata}")

        if document:
            print(f"Document preview: {document[:300]}")

        if vector is not None:
            vector_dimension = len(vector)
            vector_norm = sqrt(
                sum(float(value) * float(value) for value in vector)
            )

            first_values = [
                round(float(value), 6)
                for value in vector[:10]
            ]

            print(f"Embedding dimension: {vector_dimension}")
            print(f"Embedding norm: {vector_norm:.6f}")
            print(f"First 10 embedding values: {first_values}")

In [ ]:
import json
from services.orchestration import select_route_for_query

sample_questions = [
    "Summarize the management commentary about revenue growth and give me the relevant document context.",
    "What is the budget variance for Q3?",
    "Compare forecast versus actual revenue for North America.",
    "Give me background on the company's strategy and likely market outlook.",
]

for question in sample_questions:
    print("=" * 88)
    print(question)
    result = select_route_for_query(question)
    print(json.dumps(result, indent=2))
    print()


In [4]:
import json
import requests

question = "Summarize the relevant document context and give me the budget trend for Q3."
api_base_url = "http://127.0.0.1:8000"

print("Calling OpenAI orchestrator endpoint...")
response = requests.post(
    f"{api_base_url}/orchestrator/ask-openai",
    json={
        "user_question": question,
        "api_base_url": api_base_url,
    },
    timeout=120,
    stream=True,
)

print(f"HTTP status: {response.status_code}")
print("Content-Type:", response.headers.get("content-type"))

for line in response.iter_lines(decode_unicode=True):
    if not line:
        continue
    print(line)


Calling OpenAI orchestrator endpoint...
HTTP status: 200
Content-Type: text/event-stream; charset=utf-8
event: status
data: {"stage": "received", "message": "Request received by OpenAI Agents orchestrator.", "user_question": "Summarize the relevant document context and give me the budget trend for Q3."}
event: status
data: {"stage": "planning", "message": "Calling the OpenAI Agents SDK to select the correct tool list for the question."}
event: status
data: {"stage": "tool_selected", "message": "OpenAI Agents selected the tool list to execute.", "tools": [{"tool_name": "functions.query_finance_chunks", "path": "functions.query_finance_chunks", "method": "query_finance_chunks", "arguments": {"query": "budget trend Q3", "collection_name": "finance_docs_chunks", "n_results": 3}}, {"tool_name": "functions.list_finance_budgets", "path": "functions.list_finance_budgets", "method": "list_finance_budgets", "arguments": {"limit": 10}}]}
event: status
data: {"stage": "completed", "message": "Open

In [ ]:
# Clear / purge all existing Chroma collections
from pathlib import Path
#import chromadb
from chromadb.config import Settings

#DB_PATH = Path("C:\\Rohit\\Trainings\\repo\\FinancialAnalystCopilot\\database\\chroma").resolve()
#clientForDeletion = chromadb.PersistentClient(path=str(DB_PATH), settings=Settings(anonymized_telemetry=False))

collections = client.list_collections()
collection_names = [collection.name for collection in collections]

print(f"Found {len(collection_names)} collection(s) in: {DB_PATH}")

if collection_names:
    for name in collection_names:
        print(f"Deleting collection: {name}")
        client.delete_collection(name)

    print("All collections have been purged.")
else:
    print("No collections found to purge.")
